# Import Brent daily price data
- Author: Bryan Bravo
- Created: 2026-03-18
## Import Libraries

In [1]:
import os
import sys

# Set JAVA_HOME before importing PySpark and use findspark
os.environ['JAVA_HOME'] = r'C:\Program Files\Java\jdk-22'  # May need to remove or update in cloud environment.
import findspark
findspark.init()

import requests
import pandas as pd
import numpy as np
import json
import pyspark
from datetime import datetime as dt
from dateutil.relativedelta import relativedelta
from functools import reduce
from pyspark.sql import (
    functions as F,
    Window as W,
    types as T,
    SparkSession,
    DataFrame
)

# api keys and other hardcoded values for the Strait of Hormuz Research project.
# Note: In a production environment, these should be stored securely and not hardcoded.
import hardcoded_keys # Note: This file is added to .gitignore to prevent accidental commits of sensitive information.

### Initialize Spark Session


In [2]:
# Initialize Spark Session
spark = SparkSession.builder \
    .appName("BusinessPlanAnalysis") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.hadoop.io.native.lib.available", "false") \
    .config("spark.sql.parquet.nativeio.enabled", "false") \
    .getOrCreate()

print("Spark Session created successfully!")

Spark Session created successfully!


## Variables

In [28]:
api_key = hardcoded_keys.EIA_API_KEY
end_date = (dt.now().replace(day=1) - relativedelta(days=1)).strftime("%Y-%m-%d")

## Query

In [ ]:
# Get first batch of data = dates {'2006-01-01' through '2024-12-31'}
response = requests.get(
    "https://api.eia.gov/v2/petroleum/pri/spt/data/?frequency=daily&data[0]=value&facets[series][]=RBRTE"+
    "&start=2006-01-01&end=2024-12-31&sort[0][column]=period&sort[0][direction]=desc&offset=0&length=5000"+
    f"&api_key={api_key}"
)
eia_data1 = response.json()

# Get final batch of data {'2025-01-01' through provided value}
response = requests.get(
    "https://api.eia.gov/v2/petroleum/pri/spt/data/?frequency=daily&data[0]=value&facets[series][]=RBRTE"+
    f"&start=2025-01-01&end={end_date}&sort[0][column]=period&sort[0][direction]=desc&offset=0&length=5000"+
    f"&api_key={api_key}"
)
eia_data2 = response.json()

# Create Pandas df and union datasets.
oil_df = (
    pd.concat([
        pd.DataFrame(eia_data1['response']['data'])[['period', 'value']],
        pd.DataFrame(eia_data2['response']['data'])[['period', 'value']]
        ], axis=0, ignore_index=True)
)

# Convert to PySpark and update variable names and dtypes
oil_df = (
    spark.createDataFrame(oil_df)
    .withColumns({
        'date': F.date_format(F.to_date(F.col('period'), 'yyyy-MM-dd'), 'yyyyMMdd').cast('int'),
        'dollars_per_barrel': F.col('value').cast('double')
    })
    .select('date', 'dollars_per_barrel')
)

In [31]:
oil_df.orderBy(F.desc('date')).show()

+--------+------------------+
|    date|dollars_per_barrel|
+--------+------------------+
|20260227|             71.32|
|20260226|             71.66|
|20260225|             70.69|
|20260224|             71.21|
|20260223|              71.9|
|20260220|             72.75|
|20260219|             73.17|
|20260218|             71.78|
|20260217|             69.77|
|20260216|             70.81|
|20260213|             69.96|
|20260212|              69.8|
|20260211|             71.52|
|20260210|             71.01|
|20260209|             71.19|
|20260206|             70.45|
|20260205|             69.87|
|20260204|             71.15|
|20260203|             70.01|
|20260202|             67.72|
+--------+------------------+
only showing top 20 rows

